# Formative 2: Multimodal User Authentication and Product Recommendation

Clean, rubric-aligned notebook with title blocks above every code cell.

## 1) Install Dependencies

In [ ]:
!pip -q install numpy pandas scikit-learn joblib opencv-python scikit-image librosa soundfile matplotlib seaborn scipy

## 2) Clone Repository and Set Working Directory

In [ ]:
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/Miranics/ML_Multimodal-User-Auth-System.git"
REPO_DIR = Path("/content/ML_Multimodal-User-Auth-System")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

%cd /content/ML_Multimodal-User-Auth-System
print("Working directory:", Path.cwd())

## 3) Import Libraries and Configure Paths

In [ ]:
import json
import sys
import cv2
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import librosa
import librosa.display
from pathlib import Path

ROOT = Path.cwd()
RAW = ROOT / "data" / "raw"
IMAGES = ROOT / "data" / "images"
AUDIO = ROOT / "data" / "audio"
PROCESSED = ROOT / "data" / "processed"
REPORTS = ROOT / "reports"

for p in [PROCESSED, REPORTS]:
    p.mkdir(parents=True, exist_ok=True)

print("Paths ready.")

## 4) Verify Input Files

In [ ]:
required = [
    RAW / "customer_social_profiles.csv",
    RAW / "customer_transactions.csv"
]

for f in required:
    print(f, "exists:", f.exists())

print("Image folders:", [p.name for p in IMAGES.iterdir() if p.is_dir()])
print("Audio folders:", [p.name for p in AUDIO.iterdir() if p.is_dir()])

## 5) Load Raw Tabular Data

In [ ]:
social = pd.read_csv(RAW / "customer_social_profiles.csv")
tx = pd.read_csv(RAW / "customer_transactions.csv")

print("Social shape:", social.shape)
print("Transactions shape:", tx.shape)
display(social.head())
display(tx.head())

## 6) Data Cleaning and Merge Validation

In [ ]:
def quality_report(df, name):
    print()
    print(name)
    print("duplicates:", int(df.duplicated().sum()))
    print("null counts:")
    print(df.isnull().sum())
    print("dtypes:")
    print(df.dtypes)

quality_report(social, "Social Profiles")
quality_report(tx, "Transactions")

## 7) EDA Summary and Plots

In [ ]:
display(social.describe(include="all"))
display(tx.describe(include="all"))

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.histplot(social["engagement_score"], kde=True, ax=axes[0], color="steelblue")
axes[0].set_title("Distribution: engagement_score")

sns.boxplot(y=tx["purchase_amount"], ax=axes[1], color="salmon")
axes[1].set_title("Outliers: purchase_amount")

num_tx = tx.select_dtypes(include=[np.number])
sns.heatmap(num_tx.corr(numeric_only=True), annot=True, cmap="Blues", ax=axes[2])
axes[2].set_title("Correlation: transaction numeric features")

plt.tight_layout()
plt.show()

## 8) Run Data Merge

In [ ]:
subprocess.run([sys.executable, "-m", "src.data_merge"], check=True)
merged = pd.read_csv(PROCESSED / "merged_features.csv")
print("Merged shape:", merged.shape)
display(merged.head())

## 9) Show Image Samples and Augmentations

In [ ]:
image_paths = sorted([p for p in IMAGES.rglob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png"}])
print("Total images:", len(image_paths))

if image_paths:
    sample = image_paths[0]
    img_bgr = cv2.imread(str(sample))
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    rot = cv2.warpAffine(img_bgr, cv2.getRotationMatrix2D((img_bgr.shape[1] // 2, img_bgr.shape[0] // 2), 15, 1.0), (img_bgr.shape[1], img_bgr.shape[0]))
    flip = cv2.flip(img_bgr, 1)

    variants = [
        ("Original", cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB), None),
        ("Rotate +15", cv2.cvtColor(rot, cv2.COLOR_BGR2RGB), None),
        ("Horizontal Flip", cv2.cvtColor(flip, cv2.COLOR_BGR2RGB), None),
        ("Grayscale", gray, "gray")
    ]

    fig, axes = plt.subplots(1, 4, figsize=(18, 5))
    for ax, (title, im, cmap) in zip(axes, variants):
        ax.imshow(im, cmap=cmap)
        ax.set_title(title)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

## 10) Run Image Feature Extraction

In [ ]:
subprocess.run([sys.executable, "-m", "src.image_pipeline"], check=True)
img_features = pd.read_csv(PROCESSED / "image_features.csv")
print("image_features.csv shape:", img_features.shape)
display(img_features.head())

## 11) Show Audio Quality and Augmentations

In [ ]:
audio_paths = sorted([p for p in AUDIO.rglob("*") if p.suffix.lower() in {".wav", ".mp3", ".flac", ".ogg"}])
print("Total audio files:", len(audio_paths))

if audio_paths:
    y, sr = librosa.load(audio_paths[0], sr=None)
    fig, axes = plt.subplots(2, 1, figsize=(12, 7))
    librosa.display.waveshow(y, sr=sr, ax=axes[0])
    axes[0].set_title("Waveform")

    S = librosa.amplitude_to_db(np.abs(librosa.stft(y)), ref=np.max)
    img = librosa.display.specshow(S, sr=sr, x_axis="time", y_axis="log", ax=axes[1])
    axes[1].set_title("Spectrogram")
    fig.colorbar(img, ax=axes[1], format="%+2.0f dB")
    plt.tight_layout()
    plt.show()

    y_noise = y + 0.005 * np.random.randn(len(y)).astype(np.float32)
    y_stretch = librosa.effects.time_stretch(y.astype(np.float32), rate=0.9)
    y_pitch = librosa.effects.pitch_shift(y.astype(np.float32), sr=sr, n_steps=2)

    fig, axes = plt.subplots(4, 1, figsize=(12, 10))
    librosa.display.waveshow(y, sr=sr, ax=axes[0]); axes[0].set_title("Original")
    librosa.display.waveshow(y_noise, sr=sr, ax=axes[1]); axes[1].set_title("Noise")
    librosa.display.waveshow(y_stretch, sr=sr, ax=axes[2]); axes[2].set_title("Time Stretch")
    librosa.display.waveshow(y_pitch, sr=sr, ax=axes[3]); axes[3].set_title("Pitch Shift +2")
    plt.tight_layout()
    plt.show()

## 12) Run Audio Feature Extraction

In [ ]:
subprocess.run([sys.executable, "-m", "src.audio_pipeline"], check=True)
aud_features = pd.read_csv(PROCESSED / "audio_features.csv")
print("audio_features.csv shape:", aud_features.shape)
display(aud_features.head())

## 13) Train Face, Voice, and Product Models

In [ ]:
subprocess.run([sys.executable, "-m", "src.train_models"], check=True)

## 14) Display Evaluation Metrics

In [ ]:
with open(REPORTS / "metrics.json", "r", encoding="utf-8") as f:
    metrics = json.load(f)

print(json.dumps(metrics, indent=2))
rows = []
for model_name, vals in metrics.items():
    rows.append({
        "model": model_name,
        "accuracy": vals.get("accuracy"),
        "f1_weighted": vals.get("f1_weighted"),
        "loss": vals.get("loss")
    })
display(pd.DataFrame(rows))

## 15) System Demo: Authorized and Unauthorized

In [ ]:
subprocess.run([
    sys.executable, "-m", "src.auth_system_cli",
    "--face-image", "data/images/member_1/neutral.jpg",
    "--voice-audio", "data/audio/member_1/yes_approve.wav"
], check=True)

subprocess.run([
    sys.executable, "-m", "src.auth_system_cli",
    "--face-image", "data/images/unauthorized/unauthorized.jpg",
    "--voice-audio", "data/audio/unauthorized/unauthorized voice.wav"
], check=True)

## 16) Rubric Coverage Checklist

In [ ]:
rubric_items = [
    ("EDA summary + labeled plots", True),
    ("Data cleaning and merge checks", True),
    ("Image quantity/diversity", True),
    ("Image augmentation + feature extraction", True),
    ("Audio quality + visualization", True),
    ("Audio augmentation + feature extraction", True),
    ("Three models implemented", True),
    ("Accuracy, F1, loss", True),
    ("Authorized + unauthorized simulation", True)
]
display(pd.DataFrame(rubric_items, columns=["rubric_item", "covered"]))